I trained this file on google collab with gpu(coz normally time consuming for to train with cpu. To train all you have to do is upload processed_data.npz(213Mb),label_encoder.joblib to collab paste these code segments and run it took 52 mins to run this file)

Import Libraries & Configure GPU Hardware

In [ ]:
import math
import time
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from google.colab import files

# Select CUDA GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Hardware Accelerator: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

Set Random Seeds for Reproducibility

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True

print("Random seeds locked to 42.")

Load the Processed Dataset and Label Encoder

In [ ]:
print("Loading processed data...")
processed_data = np.load("processed_data.npz")
X_train = processed_data['X_train']
X_test = processed_data['X_test']
y_train = processed_data['y_train']
y_test = processed_data['y_test']

label_encoder = joblib.load("label_encoder.joblib")
class_names = list(label_encoder.classes_)
number_of_classes = len(class_names)
n_features = X_train.shape[1]

print(f"Loaded {len(X_train):,} training flows and {len(X_test):,} test flows with {n_features} features.")
print(f"Target Classes ({number_of_classes}): {class_names}")

Create PyTorch Datasets & GPU DataLoaders

In [ ]:
BATCH_SIZE = 1024

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

# pin_memory=True speeds up CPU-to-GPU data transfers
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"Created DataLoaders with Batch Size {BATCH_SIZE} ({len(train_loader)} batches/epoch).")

Define the FT-Transformer Neural Architecture

In [ ]:
class NumericalFeatureTokenizer(nn.Module):
    """
    Transforms continuous numerical tabular features into d-dimensional tokens:
    e_j = x_j * w_j + b_j  where w_j, b_j in R^d.
    Prepends a learnable [CLS] token at index 0.
    """
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(n_features, d_token))
        self.bias = nn.Parameter(torch.Tensor(n_features, d_token))
        self.cls_token = nn.Parameter(torch.Tensor(1, 1, d_token))
        
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.uniform_(self.bias, -1.0, 1.0)
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        x_tokens = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        return torch.cat([cls_tokens, x_tokens], dim=1)


class TransformerBlock(nn.Module):
    """
    Pre-LayerNorm Transformer Block with Multi-Head Attention and FFN.
    """
    def __init__(self, d_token: int, n_heads: int, ffn_factor: float = 2.0,
                 attention_dropout: float = 0.15, ffn_dropout: float = 0.15, residual_dropout: float = 0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.mha = nn.MultiheadAttention(embed_dim=d_token, num_heads=n_heads, dropout=attention_dropout, batch_first=True)
        self.dropout_res1 = nn.Dropout(residual_dropout)

        d_ffn = int(d_token * ffn_factor)
        self.norm2 = nn.LayerNorm(d_token)
        self.ffn = nn.Sequential(
            nn.Linear(d_token, d_ffn),
            nn.ReLU(),
            nn.Dropout(ffn_dropout),
            nn.Linear(d_ffn, d_token),
            nn.Dropout(ffn_dropout)
        )
        self.dropout_res2 = nn.Dropout(residual_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = self.norm1(x)
        attn_out, _ = self.mha(norm_x, norm_x, norm_x)
        x = x + self.dropout_res1(attn_out)
        x = x + self.dropout_res2(self.ffn(self.norm2(x)))
        return x


class FTTransformer(nn.Module):
    """
    Complete FT-Transformer: Tokenizer -> Transformer Stack -> [CLS] Head
    """
    def __init__(self, n_features: int = 69, n_classes: int = 9, d_token: int = 64,
                 n_blocks: int = 3, n_heads: int = 4, ffn_factor: float = 2.0,
                 attention_dropout: float = 0.15, ffn_dropout: float = 0.15,
                 residual_dropout: float = 0.10, head_dropout: float = 0.20, head_hidden_dim: int = 64):
        super().__init__()
        self.tokenizer = NumericalFeatureTokenizer(n_features, d_token)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_token, n_heads, ffn_factor, attention_dropout, ffn_dropout, residual_dropout)
            for _ in range(n_blocks)
        ])
        self.final_norm = nn.LayerNorm(d_token)
        self.head = nn.Sequential(
            nn.Linear(d_token, head_hidden_dim),
            nn.ReLU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden_dim, n_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.tokenizer(x)
        for block in self.blocks:
            tokens = block(tokens)
        cls_token = self.final_norm(tokens[:, 0, :])
        return self.head(cls_token)

Instantiate the Model & Check Parameters

In [ ]:
model = FTTransformer(
    n_features=n_features,
    n_classes=number_of_classes,
    d_token=64,
    n_blocks=3,
    n_heads=4,
    ffn_factor=2.0
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FT-Transformer initialized! Trainable Parameters: {total_params:,}")

Configure Balanced Loss & AdamW Optimizer

In [ ]:
# Compute balanced inverse-frequency class weights
class_counts = np.bincount(y_train, minlength=number_of_classes)
weight_values = len(y_train) / (number_of_classes * class_counts)
class_weights = torch.tensor(weight_values, dtype=torch.float32, device=device)

loss_function = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

print("Class weights applied to CrossEntropyLoss.")

Train the Model (15 Epochs)

In [ ]:
EPOCHS = 15
train_losses = []
val_losses = []
val_macro_f1s = []

start_train_time = time.time()
print(f"Training FT-Transformer for {EPOCHS} Epochs on {device}...\n" + "-"*65)

for epoch in range(EPOCHS):
    # 1. Training Phase
    model.train()
    total_train_loss = 0.0
    epoch_start = time.time()

    for features, labels in train_loader:
        features = features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(features)
        loss = loss_function(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item() * len(labels)

    avg_train_loss = total_train_loss / len(train_dataset)
    train_losses.append(avg_train_loss)

    # 2. Validation / Evaluation Phase
    model.eval()
    total_val_loss = 0.0
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)
            outputs = model(features)
            v_loss = loss_function(outputs, labels)
            total_val_loss += v_loss.item() * len(labels)
            
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(test_dataset)
    val_macro_f1 = f1_score(val_targets, val_preds, average='macro')
    
    val_losses.append(avg_val_loss)
    val_macro_f1s.append(val_macro_f1)

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro-F1: {val_macro_f1:.4f} ({epoch_time:.1f}s)")

total_mins = (time.time() - start_train_time) / 60
print(f"\nTraining Completed in {total_mins:.2f} minutes!")

Plot & Save Training Loss Curve

In [ ]:
best_epoch = int(np.argmax(val_macro_f1s)) + 1
best_f1 = max(val_macro_f1s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss Curves
ax1.plot(range(1, EPOCHS + 1), train_losses, marker='o', label='Training Loss', color='#1f77b4', linewidth=2)
ax1.plot(range(1, EPOCHS + 1), val_losses, marker='s', label='Validation Loss', color='#ff7f0e', linewidth=2)
ax1.set_title('FT-Transformer Loss Curves', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss (CrossEntropy)', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# Plot 2: Macro-F1 Curve
ax2.plot(range(1, EPOCHS + 1), val_macro_f1s, marker='^', label='Val Macro-F1', color='#2ca02c', linewidth=2)
ax2.axvline(best_epoch, color='red', linestyle='--', label=f'Best Epoch ({best_epoch})')
ax2.scatter([best_epoch], [best_f1], color='red', s=100, zorder=5)
ax2.set_title('Validation Macro-F1 per Epoch', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Macro-F1 Score', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()

# Save & Display
plt.savefig("learning_curve.png", dpi=300)
plt.show()

# Trigger automatic download
print("Downloading learning curve image...")
files.download("learning_curve.png")

Save Model Checkpoint

In [ ]:
torch.save(model.state_dict(), "ft_transformer_model.pt")
print("Model state dictionary saved as 'ft_transformer_model.pt'.")

Evaluate on the Locked Test Set (504,473 flows)

In [ ]:
print("Evaluating on Test Set...")
model.eval()
test_predictions = []
test_targets = []

start_eval_time = time.time()
with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(device)
        outputs = model(features)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_predictions.extend(preds)
        test_targets.extend(labels.numpy())

eval_time = time.time() - start_eval_time

accuracy = accuracy_score(test_targets, test_predictions)
macro_f1 = f1_score(test_targets, test_predictions, average='macro')
weighted_f1 = f1_score(test_targets, test_predictions, average='weighted')

print("\n" + "="*55)
print(f"Final Test Accuracy:    {accuracy:.4f}")
print(f"Final Test Macro-F1:    {macro_f1:.4f}")
print(f"Final Test Weighted-F1: {weighted_f1:.4f}")
print(f"Evaluation Time:        {eval_time:.2f}s ({(eval_time/len(test_targets))*1000:.4f} ms/flow)")
print("="*55 + "\n")

print("Detailed Classification Report:\n")
print(classification_report(test_targets, test_predictions, target_names=class_names, zero_division=0))

Plot & Save Confusion Matrix

In [ ]:
cm = confusion_matrix(val_targets, val_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=class_names, 
    yticklabels=class_names,
    cbar=True
)
plt.title('FT-Transformer Multiclass Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Save & Display
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

# Trigger automatic download
print("Downloading confusion matrix image...")
files.download("confusion_matrix.png")

Download Output Files

In [ ]:
# Save model weights
torch.save(model.state_dict(), "ft_transformer_model.pt")
print("Model saved as 'ft_transformer_model.pt'.")

# Trigger automatic download of model file
print("Downloading model weights...")
files.download("ft_transformer_model.pt")